# Tesla Demand Forecasting - End-to-End ML Pipeline

This notebook covers:
- Data Loading
- Data Cleaning
- Exploratory Data Analysis (EDA)
- Statistical Analysis
- Feature Engineering
- Regression Modeling
- Hyperparameter Tuning
- Feature Importance
- Time Series Forecasting
- Business Insights


## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose


## 2. Load Dataset

In [ ]:
df = pd.read_csv('tesla_sales.csv')
df.head()

## 3. Data Understanding & Cleaning

In [ ]:
print(df.shape)
print(df.info())
print(df.isnull().sum())
df = df.drop_duplicates()

## 4. Feature Engineering

In [ ]:
df['Date'] = pd.to_datetime(df['Year'].astype(str)+'-'+df['Month'].astype(str))
df['Quarter'] = df['Date'].dt.quarter
df['Month_sin'] = np.sin(2*np.pi*df['Month']/12)
df['Month_cos'] = np.cos(2*np.pi*df['Month']/12)
df['Efficiency'] = df['Estimated_Deliveries']/df['Production_Units']


## 5. Exploratory Data Analysis

In [ ]:
sns.histplot(df['Estimated_Deliveries'], kde=True)
plt.show()

numeric_df = df.select_dtypes(include=np.number)
sns.heatmap(numeric_df.corr(), annot=True)
plt.show()

## 6. Business Questions

In [ ]:
print(df.groupby('Model')['Estimated_Deliveries'].mean().sort_values(ascending=False))
print(df.groupby('Region')['Estimated_Deliveries'].sum().sort_values(ascending=False))
print(pearsonr(df['Avg_Price_USD'], df['Estimated_Deliveries']))
print(pearsonr(df['Charging_Stations'], df['Estimated_Deliveries']))

## 7. Machine Learning Pipeline

In [ ]:
target = 'Estimated_Deliveries'
df_ml = pd.get_dummies(df, columns=['Region','Model','Source_Type'], drop_first=True)
X = df_ml.drop(columns=[target,'Date'])
y = df_ml[target]

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## 8. Linear Regression

In [ ]:
lr = LinearRegression()
lr.fit(X_train,y_train)
pred = lr.predict(X_test)
print('R2:', r2_score(y_test,pred))

## 9. Random Forest + Hyperparameter Tuning

In [ ]:
params = {
 'n_estimators':[100,200],
 'max_depth':[5,10,None]
}

grid = GridSearchCV(RandomForestRegressor(random_state=42), params, cv=3)
grid.fit(X_train,y_train)
best_model = grid.best_estimator_
pred = best_model.predict(X_test)
print('MAE:', mean_absolute_error(y_test,pred))
print('RMSE:', np.sqrt(mean_squared_error(y_test,pred)))
print('R2:', r2_score(y_test,pred))

## 10. Feature Importance

In [ ]:
importance = pd.DataFrame({
 'Feature': X.columns,
 'Importance': best_model.feature_importances_
}).sort_values('Importance', ascending=False)
importance.head(20)

## 11. Time Series Forecasting Preparation

In [ ]:
monthly = df.groupby('Date')['Estimated_Deliveries'].sum().sort_index()
print('ADF p-value:', adfuller(monthly)[1])
decomposition = seasonal_decompose(monthly, model='additive', period=12)
decomposition.plot();

## 12. Final Business Recommendations
Use outputs from EDA, correlation analysis, feature importance, and forecasting to write conclusions.